# Description
This script will synthesize the data using LLM generation. It will enrich only the least frequent tags.

In [7]:
%%capture
!pip install huggingface_hub datasets google-generativeai pydantic
!pip install -U transformers accelerate bitsandbytes

In [8]:
import logging
import sys

# 1. Create a custom named logger (e.g., 'my_logger')
logger = logging.getLogger("my_logger")
logger.setLevel(logging.INFO)  # Set the minimum logging level

# 2. Check if handlers already exist to prevent duplicate output in notebooks
if not logger.handlers:
    # Create a StreamHandler to print to the console/notebook output
    console_handler = logging.StreamHandler(sys.stdout)

    # Define your custom formatting
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
    console_handler.setFormatter(formatter)

    # Add the handler to your logger
    logger.addHandler(console_handler)

# 3. Disable propagation to the root logger
logger.propagate = False

# --- Test it ---
logger.debug("This is a clean debug message from my isolated logger.")
logger.info("This is an isolated info message.")

15:00:46 | INFO | This is an isolated info message.


In [9]:
import pandas as pd
import numpy as np
import google.generativeai as genai
import json
import time
import torch
import re

from huggingface_hub import notebook_login, InferenceClient
from datasets import load_dataset, Dataset, Features, Sequence, Value, ClassLabel
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [10]:
notebook_login()

In [13]:
#----------------------------
# Load dataset & prepare metadata
#----------------------------
# MedMentions-ZS is available directly on the Hub
dataset = load_dataset("Ben10x/MedMentions-MTI881-NER")
logger.info(dataset)

all_tags_flat = [tag for sublist in dataset['train']['ner_tags'] for tag in sublist]
tag_counts = pd.Series(all_tags_flat).value_counts()

# Assuming 'tag_counts' is your Pandas Series from the EDA step
# Remove the 'O' tag as it skews the distribution
entity_counts = tag_counts.drop('O', errors='ignore')

#----------------------------
# Cutout bottom 20%
#----------------------------
# # Calculate the 20th percentile threshold
# threshold = np.percentile(entity_counts, 20)

# # Isolate the bottom 20% of classes
# bottom_20_classes = entity_counts[entity_counts <= threshold]

# print(f"Threshold for bottom 20%: {threshold} observations")
# print(f"Number of target classes: {len(bottom_20_classes)}")

# # Create a list of the specific UMLS tags we need to generate
# target_tags = bottom_20_classes.index.tolist()
# # Clean the 'B-' or 'I-' prefix to get the raw UMLS semantic type
# target_entities = list(set([tag[2:] for tag in target_tags if tag.startswith('B-') or tag.startswith('I-')]))

#----------------------------
# Cutout < 5 frequency
#----------------------------
threshold = 5
bottom_classes = entity_counts[entity_counts <= threshold]

print(f"Threshold for bottom classes: {threshold} observations")
print(f"Number of target classes: {len(bottom_classes)}")

# Create a list of the specific UMLS tags we need to generate
target_tags = bottom_classes.index.tolist()
# Clean the 'B-' or 'I-' prefix to get the raw UMLS semantic type
target_entities = list(set([tag[2:] for tag in target_tags if tag.startswith('B-') or tag.startswith('I-')]))
logger.info(f"Target Entities: {target_entities}")


15:01:23 | INFO | DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 23399
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2924
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2926
    })
})
Threshold for bottom classes: 5 observations
Number of target classes: 30
15:01:24 | INFO | Target Entities: ['T044', 'T169', 'T129', 'T167', 'T104', 'T196', 'T080', 'T039', 'T081', 'T125', 'T063', 'T026', 'T123', 'T131', 'T059', 'T121', 'T197', 'T024', 'T041', 'T019', 'T070']


In [20]:
#----------------------------
# Extract NER tags with their values
#----------------------------
target_entities_w_values = { tag: '' for tag in target_entities } # with desabreviation
target_entities_w_examples = { tag: [] for tag in target_entities } # with full sentence examples

for train_example in dataset['train']:
  full_tag_values = dict()
  last_processed_tag = ''
  for bio_tag, value in zip(train_example['ner_tags'], train_example['tokens']):
      tag = bio_tag[2:]
      if tag not in target_entities_w_values:
        last_processed_tag = tag
        continue

      # target_entities_w_examples
      target_entities_w_examples[tag] = ' '.join(train_example['tokens'])

      # target_entities_w_values
      if tag not in full_tag_values:
        logger.debug(f'Adding previously unseen {tag} for this datapoint.')
        full_tag_values[tag] = value
      elif tag in full_tag_values and tag == last_processed_tag:
        logger.debug(f'Appending {value} to {tag}')
        full_tag_values[tag] += f" {value}"
      elif tag in full_tag_values and tag != last_processed_tag:
        logger.debug(f'Duplicated {tag}, skipping')
        # full representation of tag is already present - skip
        # Unless this is done:
        # possible that the same tag will be present twice, this will 'break' the implementation
        # e.g. full_tag_values['T104'] = 'acute miocarditis acute miocarditis'
        pass
      last_processed_tag = tag


  for tag, value in full_tag_values.items():
    target_entities_w_values[tag] = value # will overwrite, if already present


logger.info(json.dumps(target_entities_w_values, indent=2))
logger.info(json.dumps(target_entities_w_examples, indent=2))

15:15:06 | INFO | {
  "T044": "Non-covalent interactions folding process processes",
  "T169": "ischemic",
  "T129": "Coxsackievirus B3 Infections",
  "T167": "human excreta and animal manure",
  "T104": "copolymers",
  "T196": "anionic",
  "T080": "UPS",
  "T039": "hydration - responsive",
  "T081": "income",
  "T125": "hypothalamic - pituitary",
  "T063": "RT-PCR ( Reverse transcription polymerase chain reaction )",
  "T026": "vacuolization",
  "T123": "neuroprotective",
  "T131": "NO\u2082",
  "T059": "Peripheral Myelin Protein 22 ( PMP22 )",
  "T121": "corticosteroid injections",
  "T197": "O\u2083",
  "T024": "slit skin smear biopsy samples",
  "T041": "sense of agency",
  "T019": "abdominal wall defect",
  "T070": "hydrogen bonding"
}
15:15:06 | INFO | {
  "T044": "Non-covalent interactions in controlling pH - responsive behaviors of self-assembled nanosystems Self-assembly and associated dynamic and reversible non-covalent interactions are the basis of protein biochemistry ( e.g

In [16]:
#----------------------------
# Loading the model
#----------------------------
model_id = "google/medgemma-1.5-4b-it"

# 1. Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" # Automatically handles GPU placement
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

In [17]:
#----------------------------
# Setting up the pipeline
#----------------------------
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    batch_size=16
)


In [18]:
#----------------------------
# Prepare prompts
#----------------------------
prompts = []
for entity in target_entities:
    description = target_entities_w_values.get(entity, "Medical Concept")

    prompt = f"""<start_of_turn>user
Generate 3 unique medical sentences. CUI: {entity}; Description: {description}.
Output strictly a JSON list:
[{{"tokens": ["token1", "token2"], "ner_tags": ["O", "B-{entity}"]}}]
Rules: Same length for tokens/tags, BIO format, individual tokens only, do not use CUI in tokens, use CUI only in ner_tags. <end_of_turn>
<start_of_turn>model
"""
    prompts.append(prompt)
logger.info(json.dumps(prompts, indent=2))

15:04:15 | INFO | [
  "<start_of_turn>user\nGenerate 3 unique medical sentences. CUI: T044; Description: Non-covalent interactions folding process processes.\nOutput strictly a JSON list:\n[{\"tokens\": [\"token1\", \"token2\"], \"ner_tags\": [\"O\", \"B-T044\"]}]\nRules: Same length for tokens/tags, BIO format, individual tokens only, do not use CUI in tokens, use CUI only in ner_tags. <end_of_turn>\n<start_of_turn>model\n",
  "<start_of_turn>user\nGenerate 3 unique medical sentences. CUI: T169; Description: ischemic.\nOutput strictly a JSON list:\n[{\"tokens\": [\"token1\", \"token2\"], \"ner_tags\": [\"O\", \"B-T169\"]}]\nRules: Same length for tokens/tags, BIO format, individual tokens only, do not use CUI in tokens, use CUI only in ner_tags. <end_of_turn>\n<start_of_turn>model\n",
  "<start_of_turn>user\nGenerate 3 unique medical sentences. CUI: T129; Description: Coxsackievirus B3 Infections.\nOutput strictly a JSON list:\n[{\"tokens\": [\"token1\", \"token2\"], \"ner_tags\": [\"

In [39]:
import os

# --- 1. Salvage Function ---
def salvage_json_objects(text):
    subtext = text.split("<unused94>")[-1]

    found_samples = []
    pattern = r'\{\s*"tokens"\s*:\s*\[.*?\]\s*,\s*"ner_tags"\s*:\s*\[.*?\]\s*\}'
    matches = re.findall(pattern, subtext, re.DOTALL)

    for match in matches:
        try:
            obj = json.loads(match)
            if (isinstance(obj.get('tokens'), list) and
                isinstance(obj.get('ner_tags'), list) and
                len(obj['tokens']) == len(obj['ner_tags'])):
                found_samples.append(obj)
        except (json.JSONDecodeError, TypeError):
            continue

    return found_samples

# --- 2. Configuration ---
CHECKPOINT_FILE = "synthetic_data_checkpoint.json"
BATCH_SIZE = 2
all_synthetic_data = []

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r") as f:
        all_synthetic_data = json.load(f)
    print(f"Resumed from checkpoint. Loaded {len(all_synthetic_data)} samples.")

# --- 3. Batched Generation with Batch-Level Error Catching ---
print(f"Starting generation for {len(prompts)} entities...")

for i in range(0, len(prompts), BATCH_SIZE):
    batch_prompts = prompts[i:i + BATCH_SIZE]

    print(f"Processing batch {i//BATCH_SIZE + 1}/{(len(prompts)-1)//BATCH_SIZE + 1}...")

    try:
        # If generation fails (e.g., OOM), the except block catches it
        results = generator(batch_prompts, max_new_tokens=1024, truncation=True)

        batch_count = 0
        for j, output in enumerate(results):
            generated_text = output[0]['generated_text']
            salvaged = salvage_json_objects(generated_text)
            all_synthetic_data.extend(salvaged)
            batch_count += len(salvaged)

        # Checkpoint: Save after every successful batch
        with open(CHECKPOINT_FILE, "w") as f:
            json.dump(all_synthetic_data, f, indent=2)

        print(f"   -> Salvaged {batch_count} samples from this batch. Total: {len(all_synthetic_data)}")

    except Exception as e:
        print(f"\nCRITICAL: Batch {i//BATCH_SIZE + 1} failed during generation!")
        print(f"Error: {e}")

        # Emergency save just in case
        with open(CHECKPOINT_FILE, "w") as f:
            json.dump(all_synthetic_data, f, indent=2)

        print("Progress saved safely to disk. Continuing to the next batch...\n")
        # Empty GPU cache if it was an OOM
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

print(f"\nSUCCESS: Finished generation process. Total samples saved: {len(all_synthetic_data)}")

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Resumed from checkpoint. Loaded 11 samples.
Starting generation for 21 entities...
Processing batch 1/11...


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   -> Salvaged 3 samples from this batch. Total: 14
Processing batch 2/11...


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

   -> Salvaged 0 samples from this batch. Total: 25
Processing batch 9/11...


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   -> Salvaged 2 samples from this batch. Total: 27
Processing batch 10/11...


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   -> Salvaged 4 samples from this batch. Total: 31
Processing batch 11/11...
   -> Salvaged 2 samples from this batch. Total: 33

SUCCESS: Finished generation process. Total samples saved: 33


In [32]:
BATCH_SIZE = 2

for i in range(0, len(prompts), BATCH_SIZE):
    if i > 0: # only 1 generation for now
      continue

    batch_prompts = prompts[i:i + BATCH_SIZE]

    print(f"Processing batch {i//BATCH_SIZE + 1}/{(len(prompts)-1)//BATCH_SIZE + 1}...")

    results = generator(batch_prompts, max_new_tokens=2048, truncation=True)

[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing batch 1/11...


In [33]:
print(results[0][0]['generated_text'])

<start_of_turn>user
Generate 3 unique medical sentences. Target Entity: T081 (income).
Output strictly a JSON list:
[{"tokens": ["token1", "token2"], "ner_tags": ["O", "B-T081"]}]
Rules: Same length for tokens/tags, BIO format, individual tokens only.<end_of_turn>
<start_of_turn>model
<unused94>thought
The user wants three unique medical sentences.
The target entity is 'T081', which represents 'income'.
The output must be a JSON list of dictionaries.
Each dictionary should have two keys: "tokens" (a list of strings) and "ner_tags" (a list of strings, with 'O' for outside and 'B-T081' for beginning of the token).
The sentences must be unique and medically relevant.
The tokens and tags must have the same length.
Each dictionary must contain only the tokens and tags for one sentence.

Let's plan the sentences:
1.  **Sentence 1:** Focus on income affecting health outcomes.
    *   Sentence: "Low income is associated with poorer health outcomes."
    *   Tokens: ["Low", "income", "is", "ass

In [13]:
# #----------------------------
# # Produce synthetic data
# #----------------------------
# print(f"Starting batched generation for {len(prompts)} entities...")
# results = generator(prompts, max_new_tokens=1024, truncation=True)

# all_synthetic_data = []

# for i, output in enumerate(results):
#     generated_text = output[0]['generated_text']
#     entity_id = target_entities[i]

#     try:
#         # Isolate the model response
#         response_part = generated_text.split("<unused94>")[-1].strip()

#         # Strip potential markdown and whitespace
#         clean_text = response_part.replace("```json", "").replace("```", "").strip()
#         if clean_text.startswith("json"):
#             clean_text = clean_text[4:].strip()

#         data = json.loads(clean_text)

#         # Align with the list of dicts format
#         valid_samples = [s for s in data if len(s.get('tokens', [])) == len(s.get('ner_tags', []))]
#         print("Adding samples:\n", valid_samples, "\n")
#         all_synthetic_data.extend(valid_samples)

#     except Exception as e:
#         print(f"Parse error for {entity_id}: {e}")
#         print(f"Raw response: {generated_text}")
#         print("-" * 50)
#         continue

# print(f"Generation Complete. Total synthetic samples: {len(all_synthetic_data)}")

[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting batched generation for 21 entities...


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Parse error for T081: Expecting value: line 1 column 1 (char 0)
Raw response: <start_of_turn>user
Generate 3 unique medical sentences. Target Entity: T081 (income).
Output strictly a JSON list:
[{"tokens": ["token1", "token2"], "ner_tags": ["O", "B-T081"]}]
Rules: Same length for tokens/tags, BIO format, individual tokens only.<end_of_turn>
<start_of_turn>model
<unused94>thought
The user wants three unique sentences, each containing the medical entity 'T081' (income).
The sentences should be medically relevant.
The output must be a JSON list of objects.
Each object should have two keys: "tokens" and "ner_tags".
The "tokens" key should contain a list of strings (the words in the sentence).
The "ner_tags" key should contain a list of strings (BIO tags for each token).
The length of the "tokens" and "ner_tags" lists must be the same.
The BIO format requires "O" for outside the entity and "B-T081" for the beginning of the entity.

Sentence 1: Focus on income affecting healthcare access.
Se

In [9]:
# #----------------------------
# # Producing synthetic data
# #----------------------------

# def generate_local_samples(entity_id, num_samples=3):
#     # Constructing the chat-formatted prompt
#     prompt = f"""<start_of_turn>user
# You are a medical data expert. Create {num_samples} unique sentences for a medical NER dataset.
# Target Entity Type: {entity_id}
# List of desabreviations for Entity Types: {str(target_entities_w_values)}

# Output ONLY a JSON list.
# Example:
# [
#   {{"tokens": ["Patient", "has", "acute", "appendicitis", "."], "ner_tags": ["O", "O", "B-{entity_id}", "I-{entity_id}", "O"]}}
# ]

# Rules:
# 1. 'tokens' and 'ner_tags' must have the same length.
# 2. Use standard BIO tagging.
# 3. Keep tokens as individual words or punctuation marks.<end_of_turn>
# <start_of_turn>model
# """

#     outputs = generator(
#         prompt,
#         max_new_tokens=1024,
#         do_sample=True,
#         temperature=0.7,
#         top_k=50,
#         top_p=0.95,
#         truncation=True
#     )

#     generated_text = outputs[0]['generated_text']

#     try:
#         # 1. Cutoff anything before last <unused95> mention
#         if "<unused95>" in generated_text:
#             generated_text = generated_text.split("<unused95>")[-1]

#         # 2. Remove the first 'json' mention (usually in ```json blocks)
#         # Also strip markdown backticks and whitespace
#         clean_text = generated_text.replace("```json", "").replace("```", "").strip()
#         if clean_text.startswith("json"):
#             clean_text = clean_text[4:].strip()

#         # 3. Parse the result as json
#         data = json.loads(clean_text)

#         # Basic validation check
#         return [s for s in data if len(s['tokens']) == len(s['ner_tags'])]
#     except Exception as e:
#         print(f"Error parsing local generation for {entity_id}: {e}")
#         print(generated_text)
#         return []

# all_synthetic_data = []
# for entity in target_entities:
#     print(f"Synthesizing data for: {entity}")
#     samples = generate_local_samples(entity)
#     print(samples)
#     all_synthetic_data.extend(samples)

# print(f"Total synthetic samples generated: {len(all_synthetic_data)}")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'top_k', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Synthesizing data for: T121


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'tokens': ['The', 'patient', 'received', 'a', 'corticosteroid', 'injection', 'for', 'inflammation', '.'], 'ner_tags': ['O', 'O', 'O', 'O', 'B-T121', 'I-T121', 'O', 'O', 'O']}, {'tokens': ['A', 'common', 'use', 'for', 'corticosteroid', 'injections', 'is', 'to', 'reduce', 'swelling', '.'], 'ner_tags': ['O', 'O', 'O', 'O', 'B-T121', 'I-T121', 'O', 'O', 'O', 'O', 'O']}, {'tokens': ['Side', 'effects', 'of', 'corticosteroid', 'injections', 'can', 'include', 'increased', 'blood', 'sugar', '.'], 'ner_tags': ['O', 'O', 'O', 'O', 'B-T121', 'I-T121', 'O', 'O', 'O', 'O', 'O']}]
Synthesizing data for: T080


...

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Error parsing local generation for T044: Unterminated string starting at: line 2 column 173 (char 174)
```json
[
  {"tokens": ["The", "folding", "process", "of", "proteins", "relies", "heavily", "on", "non-covalent", "interactions", "like", "hydrogen", "bonding", "and", "van", "der", "
[]
Synthesizing data for: T197


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'tokens': ['The', 'ozone', 'layer', 'depletion', 'is', 'a', 'major', 'environmental', 'concern', 'affecting', 'global', 'health', '.'], 'ner_tags': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}, {'tokens': ['Patients', 'with', 'asthma', 'may', 'experience', 'worsening', 'symptoms', 'during', 'high', 'ozone', 'pollution', 'events', '.'], 'ner_tags': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}]
Synthesizing data for: T125


KeyboardInterrupt: 

In [ ]:

# 3. Run Batch Generation
print(f"Starting batched generation for {len(prompts)} entities...")
# We lower max_new_tokens to 300 to prevent runaway generation
results = generator(prompts, max_new_tokens=300, truncation=True)

# 4. Process Results
all_synthetic_data = []
for i, output in enumerate(results):
    generated_text = output[0]['generated_text']
    entity_id = target_entities[i]

    try:
        # Extract JSON (re-using your parsing logic)
        clean_text = generated_text.split("<start_of_turn>model")[-1].strip()
        clean_text = clean_text.replace("```json", "").replace("```", "").strip()

        data = json.loads(clean_text)
        valid_samples = [s for s in data if len(s['tokens']) == len(s['ner_tags'])]
        all_synthetic_data.extend(valid_samples)
    except:
        print(f"Failed to parse results for {entity_id}")

print(f"Generation Complete. Total samples: {len(all_synthetic_data)}")

In [ ]:
all_synthetic_data

In [ ]:
#----------------------------
# Publishing synthetic data
#----------------------------
from datasets import load_dataset, Dataset, concatenate_datasets, Features, Sequence, ClassLabel, Value

# 1. Load the original dataset to get the schema/features
# (Assuming you are using the same dataset from your EDA)
dataset = load_dataset("Ben10x/MedMentions-MTI881-NER")
original_train = dataset['train']
ner_feature = original_train.features['ner_tags']

# 2. Convert your local 'all_synthetic_data' list to a HF Dataset
synthetic_hf = Dataset.from_list(all_synthetic_data)

# 3. Map string tags (e.g., 'B-T047') to integer IDs
# This ensures compatibility with the original MedMentions labels
def map_str_tags_to_ids(example):
    # ner_feature is a Sequence(ClassLabel), so we access .feature for mapping
    tag_names = ner_feature.feature.names
    example['ner_tags'] = [
        ner_feature.feature.str2int(tag) if tag in tag_names else 0
        for tag in example['ner_tags']
    ]
    return example

print("Mapping synthetic labels to integer IDs...")
synthetic_hf = synthetic_hf.map(map_str_tags_to_ids)

# 4. Align features and Cast
# This step ensures that types like 'int64' vs 'int32' match exactly
synthetic_hf = synthetic_hf.cast(original_train.features)

# 5. Concatenate and Shuffle
# We combine the original training data with the new synthetic observations
print(f"Original samples: {len(original_train)}")
print(f"Synthetic samples: {len(synthetic_hf)}")

enriched_train = concatenate_datasets([original_train, synthetic_hf])
enriched_train = enriched_train.shuffle(seed=42)

# 6. Update the dataset object
dataset['train'] = enriched_train

# 7. Push to the Hugging Face Hub
# Ensure you have run notebook_login() earlier in the file
repo_id = "your-hf-username/MedMentions-Enriched-NER" # Change to your repo name

dataset.push_to_hub(
    repo_id,
    private=False,
    commit_message="Enriched training data with MedGemma-4b-it synthetic samples for rare classes"
)

print(f"Successfully published enriched dataset to https://huggingface.co/datasets/{repo_id}")